In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [6]:
### Multihead Attention ###

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model # Model embedding size
        self.num_heads = num_heads # Number of parellel attention heads
        self.head_dim = d_model // num_heads # Dimensionality per head

        # Linear layers to project Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final output projection
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask = None):
        batch_size = q.size(0)

        # 1. Linear projection and split into heads
        # (batch, seq_len, d_model) -> (batch, seq_len, num_heads, head_dim)
        # then transpose into (batch, num_heads, seq_len, head_dim)
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Scaled Dot-Product Attention
        # scores shape: (batch, num_heads, seq_len, seq_len)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim))
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = F.softmax(attn_scores, dim = -1)

        # 3. Multiply by values
        # output shape: (batch, num_heads, seq_len, d_k)
        out = torch.matmul(attn_probs, V)

        # 4. Concatenate heads and project
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(out)
    
batch = 32
sequence = 10
model_dim = 512
n_heads = 8
X = torch.randn(batch, sequence, model_dim)

model = MultiHeadAttention(d_model = model_dim, num_heads = n_heads)
output = model(q = X, k = X, v = X)

print(f"Input shape:  {X.shape}")      # Expected: [32, 10, 512]
print(f"Output shape: {output.shape}") # Expected: [32, 10, 512]

# A quick sanity check: The output should have the same shape as the input
assert X.shape == output.shape, "Shape mismatch! Something is wrong."
print("\nSuccess: The model processed the input and maintained the dimensions!")


Input shape:  torch.Size([32, 10, 512])
Output shape: torch.Size([32, 10, 512])

Success: The model processed the input and maintained the dimensions!
